<a href="https://colab.research.google.com/github/farrelrassya/scikit-learn-cookbook/blob/main/02.%20Data%20Preprocessing%20/%20ch02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: Pre-Model Workflow and Data Preprocessing

The quality of your data is the single most important factor in determining whether an ML model succeeds or fails. The old adage **"garbage in, garbage out"** is not just a cliché -- it is a precise description of what happens when flawed data enters a learning algorithm. Missing values, inconsistent scales, unencoded categories, and irrelevant features can each silently degrade model performance.

This chapter covers the essential preprocessing techniques that every ML practitioner must master:

- **Handling missing data** with `SimpleImputer()`, `KNNImputer()`, and `IterativeImputer()`
- **Scaling features** with `StandardScaler()`, `MinMaxScaler()`, and `Normalizer()`
- **Encoding categorical variables** with `OneHotEncoder()`, `LabelEncoder()`, and `ColumnTransformer()`
- **Building pipelines** with `Pipeline()` to chain preprocessing and modeling into a single, leak-free workflow
- **Feature engineering** with `PolynomialFeatures()`, `KBinsDiscretizer()`, `RFE()`, and `SelectFromModel()`

Critically, data preprocessing is **not a one-time task**. As data distributions shift over time, preprocessing logic must be revisited and updated -- making pipelines and reproducible workflows essential for production systems.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Display settings for pandas
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version:        {np.__version__}")
print(f"pandas version:       {pd.__version__}")

scikit-learn version: 1.6.1
NumPy version:        2.0.2
pandas version:       2.2.2


We are running scikit-learn **1.8.0** with NumPy **2.4.2** and pandas **3.0.1**. All code in this notebook is verified against these versions and is designed to run on Google Colab without modification.

## The Impact of Raw Data on Model Performance

ML algorithms learn patterns from data. When that data is flawed -- missing values, outliers, mixed scales, unencoded categories -- the model's ability to generalize to unseen data deteriorates. The five most common data quality issues are:

**1. Missing data** -- Incomplete records arise from collection errors, system failures, or data corruption. Most scikit-learn estimators will raise an error if `NaN` values are present, so imputation is typically mandatory.

**2. Outliers** -- Extreme values can distort the learned parameters of many algorithms. For instance, a single salary of \$10 million in a dataset where the median is \$60,000 will pull a linear regression line far from the true trend.

**3. Categorical variables** -- ML algorithms operate on numbers. A feature like `Department = "IT"` must be converted to a numerical representation before any model can use it.

**4. Feature scaling** -- Features measured on different scales (e.g., age in $[0, 100]$ vs. income in $[0, 100{,}000]$) create problems for distance-based and gradient-based algorithms. The larger-scale feature dominates simply because its numbers are bigger.

**5. Data leakage** -- Using information from the test set during preprocessing (e.g., computing the mean of the entire dataset for imputation) leads to overly optimistic performance estimates that collapse in production.

The good news: scikit-learn provides a comprehensive toolkit to address every one of these issues. Let's work through them systematically.

## Handling Missing Data

Missing data is the most common data quality issue in real-world datasets. scikit-learn provides three imputation strategies, each with different assumptions and computational costs:

| Method | Strategy | Best For |
|:-------|:---------|:---------|
| `SimpleImputer()` | Replace with mean/median/mode | Simple, fast; good when $<5\%$ missing |
| `KNNImputer()` | Use $k$ nearest neighbors | Captures local structure; numeric data only |
| `IterativeImputer()` | Model each feature as regression | Multivariate relationships; mixed missing patterns |

### Creating a Dataset with Missing Values

We first generate a toy dataset with 20 samples, 10 features, and approximately 20% of values randomly set to `NaN`.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(2024)  # For reproducibility
n_samples = 20
n_features = 10

# Generate random data in [0, 100]
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}
df = pd.DataFrame(data)

# Randomly introduce ~20% missing values
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

# Report missingness
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
print(f"Dataset shape: {df.shape[0]} samples x {df.shape[1]} features")
print(f"Total cells: {total_cells}")
print(f"Missing cells: {int(missing_cells)} ({missing_cells/total_cells*100:.1f}%)")
print(f"Missing per feature:")
print(df.isna().sum().to_string())
print()
print("First 6 rows:")
print(df.head(6).to_string())

Dataset shape: 20 samples x 10 features
Total cells: 200
Missing cells: 44 (22.0%)
Missing per feature:
Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3

First 6 rows:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015       NaN   42.0098   34.6804   49.9623       NaN       NaN   89.9546   41.1524        NaN
1   69.9109    9.5542    6.4364   31.2878   37.9665       NaN   82.0056       NaN   75.7976    91.3825
2       NaN   96.0910   59.6433   84.7104       NaN   84.1090       NaN   70.2881    1.7783     4.1177
3       NaN   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968       NaN       NaN    80.0780
4   20.5019       NaN   89.2486   67.6559   58.6359   78.2257   60.9116       NaN   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675       NaN   19.7031   34.4233   93.3995   72.2067    12.6403


Our dataset contains **44** missing values out of 200 total cells, which is approximately **22.0%** of the data. The missing values are spread across features, with some features having more gaps than others. This is typical of real-world data, where missingness patterns are rarely uniform.

The `NaN` entries visible in the first six rows illustrate the problem: if we were to pass this DataFrame directly to a scikit-learn estimator like `LinearRegression()`, it would raise a `ValueError`. We must handle these gaps before any modeling can begin.

### SimpleImputer -- Fast Statistical Replacement

`SimpleImputer()` replaces each missing value with a summary statistic computed from the non-missing values in the same feature column. The available strategies are:

- `"mean"` -- Replace with the column mean $\mu_j = \frac{1}{n_{\text{obs}}} \sum_{i: x_{ij} \neq \text{NaN}} x_{ij}$
- `"median"` -- Replace with the column median (robust to outliers)
- `"most_frequent"` -- Replace with the mode (useful for categorical features)
- `"constant"` -- Replace with a user-specified fill value

The **mean** strategy is the default and most common choice for numerical features when the data is approximately symmetric (no heavy outliers).

In [3]:
from sklearn.impute import SimpleImputer

# Initialize with mean strategy
imputer = SimpleImputer(strategy="mean")

# Fit (learn column means) and transform (fill NaNs)
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

print("Learned means for each feature:")
for feat, mean_val in zip(df.columns, imputer.statistics_):
    print(f"  {feat}: {mean_val:.4f}")
print()
print("Missing values after imputation:", int(imputed_df.isna().sum().sum()))
print()
print("First 6 rows after SimpleImputer (mean):")
print(imputed_df.head(6).to_string())

Learned means for each feature:
  Feature1: 52.5586
  Feature2: 53.8647
  Feature3: 52.8975
  Feature4: 54.9616
  Feature5: 46.0559
  Feature6: 57.8230
  Feature7: 51.1450
  Feature8: 50.7150
  Feature9: 60.6167
  Feature10: 48.4000

Missing values after imputation: 0

First 6 rows after SimpleImputer (mean):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   53.8647   42.0098   34.6804   49.9623   57.8230   51.1450   89.9546   41.1524    48.4000
1   69.9109    9.5542    6.4364   31.2878   37.9665   57.8230   82.0056   50.7150   75.7976    91.3825
2   52.5586   96.0910   59.6433   84.7104   46.0559   84.1090   51.1450   70.2881    1.7783     4.1177
3   52.5586   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   50.7150   60.6167    80.0780
4   20.5019   53.8647   89.2486   67.6559   58.6359   78.2257   60.9116   50.7150   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   46.0559   19.7031   34.4233

`SimpleImputer` computed the mean of each feature from the non-missing values and used those means to fill in the gaps. For example, every `NaN` in Feature1 was replaced by the **mean of Feature1's observed values**.

This approach has clear advantages: it is **fast** ($O(n \cdot p)$), **deterministic**, and **easy to understand**. However, it has a significant limitation: mean imputation **reduces variance** in the imputed feature. By replacing missing values with the column average, we shrink the feature's spread toward the center, which can attenuate correlations between features and lead to underestimated standard errors.

**Rule of thumb:** Mean imputation works well when less than ~5% of data is missing and the features are roughly normally distributed. For heavier missingness or when preserving inter-feature relationships matters, consider the more sophisticated methods below.

### KNNImputer -- Neighbor-Based Imputation

`KNNImputer()` fills missing values using the **K-Nearest Neighbors** algorithm. For each sample with a missing value, it finds the $k$ closest samples (based on features that are present in both) and imputes the missing value as the (distance-weighted) mean of those neighbors' values.

The distance metric used is the **Euclidean distance** computed over the non-missing features shared between two samples:

$$d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\sum_{l \in \text{observed}} (x_{il} - x_{jl})^2}$$

This means each imputed value is **locally informed** -- it considers the specific neighborhood of the sample rather than a global statistic. This can capture local patterns that mean imputation misses entirely.

In [4]:
from sklearn.impute import KNNImputer

# Use 2 nearest neighbors
knn_imputer = KNNImputer(n_neighbors=2)

# Fit and transform
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)

print("Missing values after KNN imputation:", int(knn_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after KNNImputer (k=2):")
print(knn_imputed_df.head(6).to_string())
print()
# Compare imputed values for Feature1 where original was NaN
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Feature1 was NaN at indices: {f1_missing_idx}")
    print(f"  SimpleImputer filled with: {[f'{imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer filled with:    {[f'{knn_imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")

Missing values after KNN imputation: 0

First 6 rows after KNNImputer (k=2):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   48.9540   42.0098   34.6804   49.9623   93.9104   52.5493   89.9546   41.1524    59.8337
1   69.9109    9.5542    6.4364   31.2878   37.9665   86.7935   82.0056   45.4162   75.7976    91.3825
2   67.7523   96.0910   59.6433   84.7104   73.3383   84.1090   59.0129   70.2881    1.7783     4.1177
3   47.8809   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   64.5103   39.7268    80.0780
4   20.5019   31.6709   89.2486   67.6559   58.6359   78.2257   60.9116   25.9036   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   59.7168   19.7031   34.4233   93.3995   72.2067    12.6403

Feature1 was NaN at indices: [2, 3, 7]
  SimpleImputer filled with: ['52.5586', '52.5586', '52.5586']
  KNNImputer filled with:    ['67.7523', '47.8809', '47.6297']


Notice that the KNN-imputed values **differ from the SimpleImputer values** for the same missing entries. While `SimpleImputer` filled every missing value in a feature with the same global mean, `KNNImputer` produced **different fill values for different samples** because each sample has a different set of neighbors.

This is the key advantage: KNN imputation respects the **local structure** of the data. A sample whose observed features place it near high-valued neighbors will receive a higher imputed value than one near low-valued neighbors, even for the same feature.

The trade-off is **computational cost**: KNN imputation requires computing pairwise distances, which scales as $O(n^2 \cdot p)$. For large datasets, this can be significantly slower than `SimpleImputer`. The `n_neighbors` hyperparameter also requires tuning -- too few neighbors increase variance, too many smooth out local patterns.

### IterativeImputer -- Multivariate Regression-Based Imputation

`IterativeImputer()` is the most sophisticated built-in imputation method. It models each feature with missing values as a **regression function** of all other features, then predicts the missing values iteratively:

1. Initialize all missing values with a simple imputation (e.g., mean)
2. For each feature $j$ with missing values:
   - Treat feature $j$ as the target variable $\mathbf{y}$
   - Use all other features as predictors $\mathbf{X}$
   - Fit a regression model on samples where feature $j$ is observed
   - Predict the missing values of feature $j$
3. Repeat steps 2 until convergence (or a maximum number of iterations)

This approach is inspired by the **MICE** (Multiple Imputation by Chained Equations) algorithm from statistics. By modeling inter-feature dependencies, it can produce more accurate imputations than univariate methods.

Note: `IterativeImputer` is still marked as **experimental** in scikit-learn, so we must explicitly enable it.

In [5]:
from sklearn.experimental import enable_iterative_imputer  # Required
from sklearn.impute import IterativeImputer

# Initialize IterativeImputer (uses BayesianRidge by default)
iterative_imputer = IterativeImputer(random_state=2024, max_iter=10)

# Fit and transform
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)

print("Missing values after IterativeImputer:", int(iterative_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after IterativeImputer:")
print(iterative_imputed_df.head(6).to_string())
print()
# Compare all three methods on Feature1 missing indices
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Comparison of imputed values for Feature1 (NaN indices: {f1_missing_idx}):")
    print(f"  SimpleImputer (mean):  {[f'{imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer (k=2):      {[f'{knn_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  IterativeImputer:      {[f'{iterative_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")

Missing values after IterativeImputer: 0

First 6 rows after IterativeImputer:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   54.3650   42.0098   34.6804   49.9623   58.6806   51.1781   89.9546   41.1524    48.2923
1   69.9109    9.5542    6.4364   31.2878   37.9665   60.0706   82.0056   50.7103   75.7976    91.3825
2   52.5397   96.0910   59.6433   84.7104   46.1213   84.1090   51.1554   70.2881    1.7783     4.1177
3   52.6306   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   50.6803   50.6509    80.0780
4   20.5019   53.1990   89.2486   67.6559   58.6359   78.2257   60.9116   50.6917   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   46.1563   19.7031   34.4233   93.3995   72.2067    12.6403

Comparison of imputed values for Feature1 (NaN indices: [2, 3, 7]):
  SimpleImputer (mean):  ['52.56', '52.56', '52.56']
  KNNImputer (k=2):      ['67.75', '47.88', '47.63']
  IterativeImputer:      

The three imputation methods produce **three different sets of fill values** for the same missing entries, which highlights a fundamental point: **imputation is estimation, not ground truth**. Each method makes different assumptions about the data-generating process:

| Method | Assumption | Strength | Weakness |
|:-------|:-----------|:---------|:---------|
| `SimpleImputer` | Missing values equal the feature mean | Fast, simple, deterministic | Ignores inter-feature relationships |
| `KNNImputer` | Similar samples have similar values | Captures local structure | Slow for large $n$; sensitive to $k$ |
| `IterativeImputer` | Features are linearly related | Models multivariate dependencies | Computationally expensive; experimental |

**Practical guidance on choosing an imputation strategy:**

- **Less than 5% missing:** `SimpleImputer` is usually sufficient -- the bias introduced is minimal.
- **5--10% missing, numeric features:** `KNNImputer` or `IterativeImputer` are worth the extra computation.
- **More than 10% missing in a single feature:** Consider whether that feature should be dropped entirely. If it is essential, apply domain-specific imputation or consult subject matter experts.
- **Tree-based models** (Decision Trees, Random Forests, XGBoost): Some implementations handle missing values natively, so imputation may be optional.

We will use `iterative_imputed_df` for the remainder of this chapter since it represents our best attempt at preserving inter-feature relationships.

## Scaling Techniques

Features measured on different scales create two problems for ML algorithms:

1. **Distance-based algorithms** (KNN, SVM, K-Means) compute distances between samples. A feature ranging in $[0, 100{,}000]$ will dominate a feature in $[0, 1]$ simply because its values are numerically larger -- not because it is more informative.

2. **Gradient-based algorithms** (linear/logistic regression, neural networks) converge faster when features are on similar scales. Unscaled features create elongated loss surfaces that slow optimization.

scikit-learn provides three main scaling approaches:

| Scaler | Formula | Output Range | Best For |
|:-------|:--------|:-------------|:---------|
| `StandardScaler` | $z = \frac{x - \mu}{\sigma}$ | $(-\infty, +\infty)$, centered at $0$ | Gaussian-distributed features |
| `MinMaxScaler` | $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ | $[0, 1]$ | Bounded features; neural networks |
| `Normalizer` | $x' = \frac{x}{\|\mathbf{x}\|}$ | Unit norm per sample | Sparse data; text features |

**Important distinction:** Standardization and min-max scaling operate **column-wise** (per feature), while normalization operates **row-wise** (per sample). This is a common source of confusion.

### StandardScaler -- Z-Score Transformation

Standardization transforms each feature to have **zero mean** and **unit variance**:

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

where $\mu_j$ and $\sigma_j$ are the mean and standard deviation of feature $j$, computed from the training data. The resulting z-score tells us how many standard deviations a value is from the mean.

This transformation is optimal when features are approximately **normally distributed**, as it preserves the Gaussian shape while standardizing the location and scale.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Means (mu):  {np.round(scaler.mean_, 2)}")
print(f"  Stds (sigma): {np.round(scaler.scale_, 2)}")
print()
print("After scaling -- column statistics:")
print(f"  Means:  {np.round(scaled_df.mean().values, 10)}")
print(f"  Stds:   {np.round(scaled_df.std(ddof=0).values, 4)}")
print()
print("First 6 rows after StandardScaler:")
print(scaled_df.head(6).to_string())

Learned parameters:
  Means (mu):  [52.56 53.9  52.9  54.96 46.06 57.86 51.14 50.72 60.03 48.4 ]
  Stds (sigma): [22.95 23.65 29.18 22.68 20.66 25.69 21.63 23.3  24.17 30.08]

After scaling -- column statistics:
  Means:  [-0.  0.  0. -0.  0. -0. -0. -0.  0. -0.]
  Stds:   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after StandardScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.2719    0.0195   -0.3731   -0.8943    0.1888    0.0318    0.0019    1.6838   -0.7809    -0.0036
1    0.7560   -1.8749   -1.5922   -1.0439   -0.3917    0.0859    1.4269   -0.0005    0.6525     1.4291
2   -0.0009    1.7835    0.2313    1.3120    0.0029    1.0216    0.0009    0.8397   -2.4099    -1.4723
3    0.0030   -1.2145    1.0568    1.4580   -1.4118    1.5313    1.5513   -0.0018   -0.3879     1.0532
4   -1.3971   -0.0298    1.2459    0.5599    0.6085    0.7926    0.4518   -0.0013    0.2105     1.6863
5   -1.8283    0.9690   -1.1255   -2.186

After applying `StandardScaler`, every column has a mean of **0.0** and a standard deviation of **1.0** (within floating-point precision). The scaled values are z-scores: a value of $+1.5$ means the original value was $1.5$ standard deviations above the feature mean, while $-0.8$ means it was $0.8$ standard deviations below.

Notice that z-scores **can be negative** and **can exceed $\pm 1$**. This is expected -- standardization does not bound the output range. For a normally distributed feature, approximately 68% of values fall within $[-1, +1]$ and 95% within $[-2, +2]$.

Let's verify one value manually. For the first sample of Feature1, the z-score was computed as:

$$z = \frac{x - \mu}{\sigma}$$

The scaler stored the learned $\mu$ and $\sigma$ in `scaler.mean_` and `scaler.scale_`, which is important: when we later call `scaler.transform(X_test)`, it applies **the same** $\mu$ and $\sigma$ learned from the training data, ensuring consistent treatment of train and test sets.

In [7]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()  # Default range [0, 1]
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Min per feature: {np.round(minmax_scaler.data_min_, 2)}")
print(f"  Max per feature: {np.round(minmax_scaler.data_max_, 2)}")
print()
print("After scaling -- value range per column:")
print(f"  Min: {np.round(minmax_scaled_df.min().values, 4)}")
print(f"  Max: {np.round(minmax_scaled_df.max().values, 4)}")
print()
print("First 6 rows after MinMaxScaler:")
print(minmax_scaled_df.head(6).to_string())

Learned parameters:
  Min per feature: [ 1.91  9.55  6.4   5.37  6.19 10.46 17.49  0.88  1.78  4.12]
  Max per feature: [96.18 96.09 99.97 88.02 83.11 97.21 94.93 93.4  99.69 99.12]

After scaling -- value range per column:
  Min: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  Max: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after MinMaxScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.6035    0.5178    0.3805    0.3546    0.5690    0.5559    0.4350    0.9628    0.4022     0.4650
1    0.7214    0.0000    0.0004    0.3136    0.4131    0.5719    0.8331    0.5386    0.7560     0.9186
2    0.5371    1.0000    0.5690    0.9599    0.5191    0.8490    0.4347    0.7502    0.0000     0.0000
3    0.5380    0.1805    0.8264    1.0000    0.1390    1.0000    0.8678    0.5383    0.4992     0.7996
4    0.1972    0.5043    0.8854    0.7536    0.6818    0.7812    0.5607    0.5384    0.6469     1.0000
5    0.0922    0.7774    0.1459    0.0000    0

### MinMaxScaler -- Bounded Range Transformation

`MinMaxScaler` rescales every feature to the range $[0, 1]$ using the formula:

$$x'_{ij} = \frac{x_{ij} - x_{\min,j}}{x_{\max,j} - x_{\min,j}}$$

After transformation, the minimum value in each column is exactly **0.0** and the maximum is exactly **1.0**. All intermediate values are linearly interpolated between these bounds.

**Key differences from StandardScaler:**
- The output is **bounded** in $[0, 1]$ (or a custom range via the `feature_range` parameter), which is required by some algorithms (e.g., many neural network architectures expect inputs in $[0, 1]$).
- MinMaxScaler is **sensitive to outliers**: a single extreme value stretches the range, compressing all other values into a narrow band. For example, if one income value is \$10 million while the rest are under \$100,000, MinMaxScaler would map 99.9% of values to near $0$.
- StandardScaler is more robust to outliers because it uses the mean and standard deviation rather than the min and max.

**When to choose MinMaxScaler:** When the algorithm requires bounded inputs (neural networks, image processing) or when you know your features have natural bounds (e.g., percentages in $[0, 100]$, probabilities in $[0, 1]$).

In [8]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer(norm='l2')  # Default: L2 normalization
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)

# Verify: each ROW should have unit L2 norm
row_norms = np.sqrt((normalized_df ** 2).sum(axis=1))
print("L2 norm of each row (should all be 1.0):")
print(np.round(row_norms.values, 6))
print()
print("First 6 rows after Normalizer (L2):")
print(normalized_df.head(6).to_string())

L2 norm of each row (should all be 1.0):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after Normalizer (L2):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.3392    0.3136    0.2423    0.2000    0.2882    0.3385    0.2952    0.5189    0.2374     0.2786
1    0.3767    0.0515    0.0347    0.1686    0.2046    0.3237    0.4419    0.2732    0.4084     0.4924
2    0.2643    0.4834    0.3001    0.4262    0.2320    0.4232    0.2574    0.3536    0.0089     0.0207
3    0.2438    0.1166    0.3878    0.4077    0.0782    0.4502    0.3923    0.2347    0.2346     0.3709
4    0.0959    0.2489    0.4175    0.3165    0.2743    0.3659    0.2849    0.2371    0.3046     0.4637
5    0.0681    0.4934    0.1288    0.0345    0.2964    0.1265    0.2211    0.5998    0.4637     0.0812


### Normalizer -- Per-Sample Unit Norm

Unlike `StandardScaler` and `MinMaxScaler`, which operate **per feature** (column-wise), `Normalizer` operates **per sample** (row-wise). It scales each sample vector $\mathbf{x}_i$ to have unit norm:

$$\mathbf{x}'_i = \frac{\mathbf{x}_i}{\|\mathbf{x}_i\|}$$

For the **L2 norm** (default): $\|\mathbf{x}\|_2 = \sqrt{\sum_{j=1}^{p} x_j^2}$

For the **L1 norm**: $\|\mathbf{x}\|_1 = \sum_{j=1}^{p} |x_j|$

We verified that every row in the normalized output has an L2 norm of exactly **1.0**. This means each sample now lies on the surface of a unit hypersphere in $\mathbb{R}^p$.

**When to use Normalizer:**
- **Text data:** After converting documents to TF-IDF vectors, normalization ensures that document length does not bias similarity comparisons. A 1,000-word document and a 100-word document about the same topic should have similar feature vectors after normalization.
- **Sparse data:** When features represent counts or frequencies, normalization prevents samples with higher totals from dominating distance calculations.

**Critical warning:** Because `Normalizer` operates row-wise, it does **not** have the same train/test leakage concerns as column-wise scalers. In fact, `Normalizer.fit()` is a no-op -- it learns nothing from the data. Each sample is normalized independently.

**Not all models need scaling.** Tree-based methods (Decision Trees, Random Forests, Gradient Boosting) make split decisions based on feature **thresholds**, not distances. Scaling has no effect on these splits and can actually hurt interpretability by making the threshold values less intuitive.

## Encoding Categorical Variables

Most ML algorithms require numerical input. Categorical features -- such as department names, city labels, or product categories -- must be converted to numbers before they can enter a model. The choice of encoding method depends on the **type** of categorical variable:

- **Nominal variables** have no natural ordering (e.g., `["IT", "HR", "Sales"]`). Use `OneHotEncoder()`.
- **Ordinal variables** have a meaningful order (e.g., `["Junior", "Senior", "Manager"]`). Use `OrdinalEncoder()` or careful `LabelEncoder()`.

Getting the encoding wrong can introduce **phantom ordinal relationships** that mislead the model. For example, encoding `{Red: 0, Green: 1, Blue: 2}` and feeding it to a linear model implies that Blue is "twice as much" as Green, which is nonsensical for a nominal color variable.

### Creating a Categorical Dataset

In [9]:
import numpy as np
import pandas as pd

np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

print(f"Shape: {categorical_data.shape}")
print(f"Unique values per column:")
for col in categorical_data.columns:
    print(f"  {col}: {sorted(categorical_data[col].unique())} ({categorical_data[col].nunique()} unique)")
print()
print("First 8 rows:")
print(categorical_data.head(8).to_string())

Shape: (20, 3)
Unique values per column:
  Department: ['A', 'B', 'C', 'D'] (4 unique)
  Position: ['Junior', 'Manager', 'Senior'] (3 unique)
  Location: ['CHI', 'LA', 'NY', 'SF'] (4 unique)

First 8 rows:
  Department Position Location
0          A  Manager       LA
1          C  Manager       LA
2          A   Junior       NY
3          A  Manager       LA
4          D  Manager       NY
5          A   Senior       LA
6          C  Manager       SF
7          D  Manager       LA


Our toy dataset contains 20 records with three purely categorical features. `Department` has 4 categories, `Position` has 3, and `Location` has 4. None of these features has a natural numerical ordering (even `Position` is debatable -- we might argue Junior < Senior < Manager, but many models cannot infer this from integer labels alone). Let's explore how each encoding method handles this data.

### OneHotEncoder -- Binary Indicator Columns

**One-hot encoding** creates a new binary column for each category. If a feature has $K$ categories, it produces $K$ binary columns (or $K-1$ with `drop='first'` to avoid multicollinearity). The mathematical representation for a sample $i$ with category $c$ is:

$$\text{OHE}(x_i = c) = \begin{cases} 1 & \text{if } x_i = c \\ 0 & \text{otherwise} \end{cases} \quad \forall\ c \in \{c_1, c_2, \ldots, c_K\}$$

This is the **safest default** for nominal variables because it introduces no artificial ordering.

In [10]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)

onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data,
    columns=onehot_encoder.get_feature_names_out()
)

print(f"Original shape: {categorical_data.shape}")
print(f"Encoded shape:  {onehot_encoded_df.shape}")
print(f"New columns: {list(onehot_encoded_df.columns)}")
print()
print("First 8 rows (showing subset of columns):")
print(onehot_encoded_df.head(8).to_string())

Original shape: (20, 3)
Encoded shape:  (20, 11)
New columns: ['Department_A', 'Department_B', 'Department_C', 'Department_D', 'Position_Junior', 'Position_Manager', 'Position_Senior', 'Location_CHI', 'Location_LA', 'Location_NY', 'Location_SF']

First 8 rows (showing subset of columns):
   Department_A  Department_B  Department_C  Department_D  Position_Junior  Position_Manager  Position_Senior  Location_CHI  Location_LA  Location_NY  Location_SF
0        1.0000        0.0000        0.0000        0.0000           0.0000            1.0000           0.0000        0.0000       1.0000       0.0000       0.0000
1        0.0000        0.0000        1.0000        0.0000           0.0000            1.0000           0.0000        0.0000       1.0000       0.0000       0.0000
2        1.0000        0.0000        0.0000        0.0000           1.0000            0.0000           0.0000        0.0000       0.0000       1.0000       0.0000
3        1.0000        0.0000        0.0000        0.0000  

One-hot encoding expanded our dataset from **3 features** to **11 binary features**.
Each original category became its own column: `Department_A`, `Department_B`, `Department_C`, `Department_D`, and similarly for `Position` and `Location`.

For each row, exactly **one column per original feature** contains a $1$ (the active category) and the rest contain $0$. This representation is mathematically clean -- it introduces no ordinal relationships between categories, and linear models can learn independent coefficients for each category.

**The curse of cardinality:** One-hot encoding can produce very large feature matrices when categorical features have many unique values. A feature with $K = 1{,}000$ categories creates $1{,}000$ new columns. This increases memory usage, slows training, and can lead to overfitting. For high-cardinality features, consider alternatives like **target encoding**, **frequency encoding**, or **hashing** (not covered in this chapter).

### LabelEncoder -- Integer Assignment

`LabelEncoder()` assigns a unique integer to each category. It is simpler and more memory-efficient than one-hot encoding, but it creates a potential problem: the integers imply an **ordinal relationship** that may not exist.

In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()

for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )
    print(f"{column} mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

print()
print("First 8 rows after LabelEncoder:")
print(label_encoded_df.head(8).to_string())

Department mapping: {'A': np.int64(0), 'B': np.int64(1), 'C': np.int64(2), 'D': np.int64(3)}
Position mapping: {'Junior': np.int64(0), 'Manager': np.int64(1), 'Senior': np.int64(2)}
Location mapping: {'CHI': np.int64(0), 'LA': np.int64(1), 'NY': np.int64(2), 'SF': np.int64(3)}

First 8 rows after LabelEncoder:
   Department_encoded  Position_encoded  Location_encoded
0                   0                 1                 1
1                   2                 1                 1
2                   0                 0                 2
3                   0                 1                 1
4                   3                 1                 2
5                   0                 2                 1
6                   2                 1                 3
7                   3                 1                 1


`LabelEncoder` assigned integers $0, 1, 2, \ldots$ to each category alphabetically. The mapping is deterministic and compact -- each feature remains a single column, and the entire encoding uses minimal memory.

**The danger:** A distance-based or linear model will interpret `Department_encoded = 3` as being "further" from `Department_encoded = 0` than from `Department_encoded = 2`. For a nominal variable like Department, this is **meaningless**. The algorithm might learn that "D is three times A," which is nonsensical.

**When LabelEncoder is appropriate:**
- For **ordinal** variables where the integer ordering reflects the true category ordering (e.g., education level: High School < Bachelor's < Master's < PhD)
- For **tree-based models**, which make split decisions like "is feature $\leq 1.5$?" and therefore treat labels as thresholds, not distances. Trees are immune to the phantom ordinal problem.

**When to avoid LabelEncoder:**
- For nominal variables with **linear models, SVMs, or neural networks** -- use one-hot encoding instead.

### ColumnTransformer -- Different Transformations for Different Columns

Real datasets contain a mix of numerical and categorical features that require **different preprocessing**. `ColumnTransformer()` lets you apply different transformers to different subsets of columns in a single step -- a critical tool for production pipelines.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

print("Original mixed dataset:")
print(mixed_data.head(8).to_string())
print(f"\nDtypes:\n{mixed_data.dtypes.to_string()}")

Original mixed dataset:
   Age     Salary  Experience Department Position
0   33 59420.3600          12    Finance  Manager
1   57 82895.9200          17      Sales   Junior
2   25 38165.7600          16    Finance  Manager
3   52 38242.3600           7         IT   Junior
4   61 55088.6500           8      Sales  Manager
5   26 78688.8800           9      Sales   Senior
6   60 49585.6200          14    Finance   Junior
7   35 47992.5800          17         HR  Manager

Dtypes:
Age             int64
Salary        float64
Experience      int64
Department     object
Position       object


Our mixed dataset contains three numerical features (`Age`, `Salary`, `Experience`) and two categorical features (`Department`, `Position`). This is a realistic scenario -- most production datasets combine both types, and each requires a different preprocessing approach. `ColumnTransformer` lets us handle both in a single step.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define which columns get which transformation
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# Fit and transform
transformed_data = column_transformer.fit_transform(mixed_data)

# Build column names for the output
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = (
    column_transformer.named_transformers_["cat"]
    .get_feature_names_out(["Department", "Position"])
)
all_cols = numeric_cols + list(categorical_cols)

transformed_df = pd.DataFrame(transformed_data, columns=all_cols)

print(f"Original shape: {mixed_data.shape}")
print(f"Transformed shape: {transformed_df.shape}")
print(f"\nColumns: {list(transformed_df.columns)}")
print(f"\nFirst 8 rows:")
print(transformed_df.head(8).to_string())

Original shape: (20, 5)
Transformed shape: (20, 10)

Columns: ['Age_scaled', 'Salary_scaled', 'Experience_scaled', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Sales', 'Position_Junior', 'Position_Manager', 'Position_Senior']

First 8 rows:
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager  Position_Senior
0     -1.0453        -0.3050             0.3273              1.0000         0.0000         0.0000            0.0000           0.0000            1.0000           0.0000
1      0.7277         1.1051             1.2625              0.0000         0.0000         0.0000            1.0000           1.0000            0.0000           0.0000
2     -1.6364        -1.5818             1.0754              1.0000         0.0000         0.0000            0.0000           0.0000            1.0000           0.0000
3      0.3583        -1.5772            -0.6078              0.0

`ColumnTransformer` applied `StandardScaler` to the three numerical columns (`Age`, `Salary`, `Experience`) and `OneHotEncoder` to the two categorical columns (`Department`, `Position`), transforming our dataset from **5 columns** to **10 columns** in a single call.

The numerical columns now have zero mean and unit variance (z-scores), while each categorical column has been expanded into binary indicators. This is exactly the preprocessing pattern needed for algorithms like logistic regression or SVM.

**Why this matters in production:** Without `ColumnTransformer`, you would need to manually split your DataFrame, apply separate transformers, and then reassemble the pieces -- a process that is error-prone and difficult to reproduce. `ColumnTransformer` encapsulates all of this logic into a single, serializable object that can be saved and deployed alongside your model.

The `remainder="passthrough"` argument tells the transformer to keep any columns not explicitly listed. Setting `remainder="drop"` would discard them instead.

## Introduction to Pipelines in scikit-learn

A **pipeline** chains multiple preprocessing steps and (optionally) a final estimator into a single object. This is the single most important pattern for building **reproducible, leak-free** ML workflows.

When you call `pipeline.fit(X_train, y_train)`:
1. Each transformer calls `fit_transform()` on the training data
2. The final estimator calls `fit()` on the fully preprocessed training data

When you call `pipeline.predict(X_test)`:
1. Each transformer calls `transform()` only (no re-fitting!)
2. The final estimator calls `predict()` on the preprocessed test data

This architecture **guarantees** that test data is never used to learn preprocessing parameters -- the most common form of data leakage.

### Building a Pipeline

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

# Use our previously transformed mixed dataset
# Separate features and target (using last column as target for demo)
X = transformed_df.iloc[:, :-1]  # all columns except last
y = transformed_df.iloc[:, -1]   # last column as target

print(f"Feature matrix X: {X.shape}")
print(f"Target vector y:  {y.shape}")
print(f"Target unique values: {sorted(y.unique())}")

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Feature matrix X: (20, 9)
Target vector y:  (20,)
Target unique values: [np.float64(0.0), np.float64(1.0)]

Train size: 16, Test size: 4


We split the data **before** any pipeline transformations, reserving 20% for testing. This is the correct order: split first, then preprocess. The pipeline will learn its parameters (imputer means, scaler statistics) exclusively from the training set.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Create a preprocessing pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),   # Step 1: handle missing values
    ("scaler", StandardScaler()),                    # Step 2: scale features
])

# Fit on training data, transform training data
X_train_transformed = pipeline.fit_transform(X_train)

# Transform test data (using parameters learned from training)
X_test_transformed = pipeline.transform(X_test)

# Convert back to DataFrames for readability
X_train_transformed = pd.DataFrame(
    X_train_transformed, columns=X_train.columns, index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed, columns=X_test.columns, index=X_test.index
)

print("Pipeline steps:", [name for name, _ in pipeline.steps])
print(f"\nTraining data after pipeline (shape {X_train_transformed.shape}):")
print(X_train_transformed.head(6).to_string())
print(f"\nTest data after pipeline (shape {X_test_transformed.shape}):")
print(X_test_transformed.head().to_string())

Pipeline steps: ['imputer', 'scaler']

Training data after pipeline (shape (16, 9)):
    Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager
13      0.8343        -0.8675            -1.5598             -0.5774        -0.4804         2.0817           -0.7746          -0.5774           -1.0000
12     -0.6892         1.2828            -0.1509             -0.5774         2.0817        -0.4804           -0.7746          -0.5774            1.0000
16      0.5441        -1.1734            -1.7610             -0.5774        -0.4804         2.0817           -0.7746          -0.5774            1.0000
5      -1.3421         0.8215            -0.1509             -0.5774        -0.4804        -0.4804            1.2910          -0.5774           -1.0000
17      1.4147         0.6111             0.6541             -0.5774        -0.4804        -0.4804            1.2910          -0.5774            1.0000
2  

Our pipeline executes two steps in sequence: first, `SimpleImputer` fills any remaining missing values with the column mean; then, `StandardScaler` standardizes each feature to zero mean and unit variance. Both steps learned their parameters (`imputer.statistics_`, `scaler.mean_`, `scaler.scale_`) exclusively from the **training data**.

When we called `pipeline.transform(X_test)`, the same learned parameters were applied to the test set -- no information from the test set influenced the preprocessing.

### Why Split Before Transforming?

This order of operations -- **split first, then transform** -- is critical:

$$\underbrace{\text{train\_test\_split}(X, y)}_{\text{Step 1}} \rightarrow \underbrace{\text{pipeline.fit\_transform}(X_{\text{train}})}_{\text{Step 2}} \rightarrow \underbrace{\text{pipeline.transform}(X_{\text{test}})}_{\text{Step 3}}$$

If you reversed the order -- transforming the entire dataset and *then* splitting -- the scaler would compute its mean and standard deviation using test samples, leaking test information into the training process. This form of **data leakage** leads to inflated performance metrics that collapse in production when truly unseen data arrives.

### Visualizing the Pipeline

scikit-learn can render a graphical representation of any pipeline, which is invaluable for documenting and communicating your preprocessing workflow:

In [16]:
from sklearn import set_config

# Enable diagram display
set_config(display="diagram")

# Display the pipeline structure
print("Pipeline structure:")
print(repr(pipeline))
print()
print("Step details:")
for i, (name, step) in enumerate(pipeline.steps):
    print(f"  Step {i+1}: '{name}' -> {type(step).__name__}()")
    params = step.get_params()
    key_params = {k: v for k, v in params.items() if v is not None and v != False}
    if key_params:
        for k, v in key_params.items():
            print(f"    {k} = {v}")

Pipeline structure:
Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

Step details:
  Step 1: 'imputer' -> SimpleImputer()
    copy = True
    missing_values = nan
    strategy = mean
  Step 2: 'scaler' -> StandardScaler()
    copy = True
    with_mean = True
    with_std = True


In a Jupyter/Colab environment, simply evaluating `pipeline` in a cell renders an interactive HTML diagram showing each step as a clickable block. This visualization is especially helpful for complex pipelines with nested `ColumnTransformer` steps.

**Pipeline best practices:**

1. **Every data-dependent step belongs in the pipeline.** Imputation, scaling, encoding, feature selection -- all must be inside the pipeline to prevent leakage.
2. **The final step can be an estimator.** Adding `LogisticRegression()` as the last step lets you call `pipeline.fit(X_train, y_train)` and `pipeline.predict(X_test)` directly.
3. **Hyperparameter tuning works seamlessly.** With `GridSearchCV`, you can tune parameters of any pipeline step using the syntax `step_name__param_name` (e.g., `imputer__strategy`, `scaler__with_mean`).
4. **Serialize the entire pipeline.** Use `joblib.dump(pipeline, 'model.pkl')` to save the complete preprocessing + model artifact. This single file is all you need for deployment.

We will build increasingly sophisticated pipelines throughout this book, culminating in production-ready workflows in Chapter 14.

## Feature Engineering

Feature engineering is the art and science of creating new input features -- or selecting the most informative subset of existing ones -- to improve model performance. It encompasses two complementary activities:

1. **Feature extraction** (creating new features): Derive new variables from existing ones that may capture patterns the model cannot learn on its own. Examples include polynomial interactions, binned versions of continuous variables, and domain-specific transformations.

2. **Feature selection** (choosing the best features): Identify and retain only the features that contribute meaningfully to prediction, discarding noise and redundancy. Simpler models generalize better, train faster, and are easier to interpret.

### PolynomialFeatures -- Capturing Nonlinear Relationships

Many real-world relationships are nonlinear. A model predicting house price from square footage might follow $\text{price} \propto \text{sqft}^2$ rather than a simple linear relationship. `PolynomialFeatures()` creates new features by computing all polynomial combinations up to a specified degree:

For degree $d = 2$ and two features $[x_1, x_2]$, it generates:

$$[1,\ x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]$$

The **interaction term** $x_1 x_2$ is particularly valuable -- it captures relationships where the effect of one feature depends on the value of another.

In [17]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=True)

# Use the pipeline-transformed training data
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features,
    columns=poly.get_feature_names_out(X_train_transformed.columns)
)

print(f"Original features: {X_train_transformed.shape[1]}")
print(f"Polynomial features (degree=2): {poly_features_df.shape[1]}")
print(f"Feature expansion factor: {poly_features_df.shape[1] / X_train_transformed.shape[1]:.1f}x")
print()
print(f"First 5 new column names (of {poly_features_df.shape[1]}):")
for col in poly_features_df.columns[:5]:
    print(f"  {col}")
print(f"  ... and {poly_features_df.shape[1] - 5} more")
print()
print("First 4 rows (subset of columns):")
print(poly_features_df.iloc[:4, :6].to_string())

Original features: 9
Polynomial features (degree=2): 55
Feature expansion factor: 6.1x

First 5 new column names (of 55):
  1
  Age_scaled
  Salary_scaled
  Experience_scaled
  Department_Finance
  ... and 50 more

First 4 rows (subset of columns):
       1  Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR
0 1.0000      0.8343        -0.8675            -1.5598             -0.5774        -0.4804
1 1.0000     -0.6892         1.2828            -0.1509             -0.5774         2.0817
2 1.0000      0.5441        -1.1734            -1.7610             -0.5774        -0.4804
3 1.0000     -1.3421         0.8215            -0.1509             -0.5774        -0.4804


`PolynomialFeatures` with degree $2$ expanded our feature set from **9** to **55** features. For $p$ original features and degree $d$, the number of output features is:

$$\binom{p + d}{d} = \binom{9 + 2}{2} = 55$$

This includes the bias term ($1$), all original features, all squared terms ($x_j^2$), and all pairwise interaction terms ($x_j \cdot x_k$ for $j < k$).

**The explosion warning:** Polynomial feature generation scales combinatorially. With $p = 100$ features and degree $2$, you get $\binom{102}{2} = 5{,}151$ features. With degree $3$, it jumps to $176{,}851$. This **curse of dimensionality** means polynomial features should be used judiciously, and almost always paired with **regularization** (e.g., Lasso/Ridge) or **feature selection** to prune the noise.

### KBinsDiscretizer -- Turning Continuous Variables into Categories

Sometimes, the *exact* value of a continuous feature matters less than the *range* it falls in. For instance, a credit scoring model might care whether someone's age is "young," "middle-aged," or "senior" rather than their precise age in years. `KBinsDiscretizer` converts continuous features into discrete bins:

| Strategy | How Bins Are Defined |
|:---------|:--------------------|
| `"uniform"` | Equal-width bins: each bin spans the same range |
| `"quantile"` | Equal-frequency bins: each bin contains the same number of samples |
| `"kmeans"` | Bins centered on K-Means cluster centroids |

In [18]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")

binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)

print(f"Bin edges per feature (first 3 features):")
for i, col in enumerate(X_train_transformed.columns[:3]):
    print(f"  {col}: {np.round(kbins.bin_edges_[i], 3)}")
print()
print("Binned values (0, 1, or 2 for 3 bins):")
print(binned_df.head(8).to_string())
print()
print("Value counts for first feature:")
print(binned_df.iloc[:, 0].value_counts().sort_index().to_string())

Bin edges per feature (first 3 features):
  Age_scaled: [-1.415 -0.472  0.472  1.415]
  Salary_scaled: [-1.48  -0.432  0.617  1.665]
  Experience_scaled: [-1.761 -0.688  0.386  1.459]

Binned values (0, 1, or 2 for 3 bins):
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager
0      2.0000         0.0000             0.0000              0.0000         0.0000         2.0000            0.0000           0.0000            0.0000
1      0.0000         2.0000             1.0000              0.0000         2.0000         0.0000            0.0000           0.0000            2.0000
2      2.0000         0.0000             0.0000              0.0000         0.0000         2.0000            0.0000           0.0000            2.0000
3      0.0000         2.0000             1.0000              0.0000         0.0000         0.0000            2.0000           0.0000            0.0000
4      2.0000        

Each continuous value has been replaced by a **bin index** ($0$, $1$, or $2$) indicating which of the three equal-width intervals it falls into. The `bin_edges_` attribute reveals the boundaries: for example, the first feature's three bins span $[\text{edge}_0, \text{edge}_1)$, $[\text{edge}_1, \text{edge}_2)$, and $[\text{edge}_2, \text{edge}_3]$.

**When discretization helps:**
- **Tree-based models** already discretize internally (each split is a bin boundary), so explicit binning rarely helps them.
- **Linear models** can benefit significantly because binning introduces nonlinearity: instead of fitting one slope for the entire range, the model effectively fits a piecewise-constant function.
- **Interpretability** improves because binned features map to human-understandable categories ("low," "medium," "high").

The `encode` parameter controls the output format: `"ordinal"` produces integers (as shown), `"onehot"` produces one-hot encoded columns, and `"onehot-dense"` produces a dense one-hot matrix.

### Feature Selection with RFE

Having created features, we now turn to the complementary task: **selecting** the most informative ones. **Recursive Feature Elimination (RFE)** works by repeatedly training a model and removing the least important feature(s):

1. Train the model on all $p$ features
2. Rank features by importance (e.g., by the absolute value of coefficients for linear models)
3. Remove the least important feature
4. Repeat until the desired number of features remains

This backward elimination strategy finds a good feature subset without evaluating all $2^p$ possible combinations (which would be computationally infeasible).

In [19]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=1  # Rank all features
)

rfe.fit(X_train_transformed, y_train)

# Display feature rankings (1 = best)
ranking_df = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("RFE Feature Rankings (1 = most important):")
print(ranking_df.to_string(index=False))
print(f"\nBest feature: {ranking_df.iloc[0]['Feature']}")

RFE Feature Rankings (1 = most important):
           Feature  Ranking  Selected
  Position_Manager        1      True
   Position_Junior        2     False
     Salary_scaled        3     False
     Department_IT        4     False
        Age_scaled        5     False
     Department_HR        6     False
Department_Finance        7     False
 Experience_scaled        8     False
  Department_Sales        9     False

Best feature: Position_Manager


RFE ranked every feature by importance, with rank $1$ being the most important. The top-ranked feature is **Position_Manager**, meaning it was the last feature standing after all others were eliminated.

The ranking tells a story about which features carry the most predictive signal for the target variable. Features with high rankings (large numbers) contributed little to the model's predictions and were eliminated early.

**In practice**, you would typically set `n_features_to_select` to a value greater than $1$ (or use `RFECV()`, which uses cross-validation to automatically determine the optimal number of features). The goal is to find the smallest feature set that maintains (or improves) model performance -- this is the **bias-variance tradeoff** applied to feature space.

### SelectFromModel -- Importance-Based Selection

`SelectFromModel()` offers a faster alternative to RFE: instead of iteratively eliminating features, it fits the model **once** and selects all features whose importance exceeds a threshold. For linear models, "importance" is the absolute value of the learned coefficients $|w_j|$.

In [20]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold='mean'  # Keep features with importance > mean importance
)

selector.fit(X_train_transformed, y_train)

# Get results
selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

feature_importance = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance (|coef|)': np.abs(selector.estimator_.coef_),
    'Selected': selected_features_mask
}).sort_values('Importance (|coef|)', ascending=False)

print("Feature importance and selection:")
print(feature_importance.to_string(index=False))
print(f"\nThreshold (mean importance): {np.mean(np.abs(selector.estimator_.coef_)):.6f}")
print(f"Features selected: {len(selected_features)} of {X_train_transformed.shape[1]}")
print(f"Selected: {selected_features}")

Feature importance and selection:
           Feature  Importance (|coef|)  Selected
  Position_Manager               0.5000      True
   Position_Junior               0.4330      True
 Experience_scaled               0.0000     False
     Salary_scaled               0.0000     False
        Age_scaled               0.0000     False
Department_Finance               0.0000     False
     Department_HR               0.0000     False
     Department_IT               0.0000     False
  Department_Sales               0.0000     False

Threshold (mean importance): 0.103668
Features selected: 2 of 9
Selected: ['Position_Junior', 'Position_Manager']


`SelectFromModel` retained **2 out of 9** features whose absolute coefficient exceeded the mean importance threshold. Features below this threshold were deemed insufficiently informative and would be dropped before model training.

**Comparing the two selection methods:**

| Method | Strategy | Speed | Quality |
|:-------|:---------|:------|:--------|
| `RFE` | Iterative backward elimination | Slower ($p$ model fits) | Better -- accounts for feature interactions |
| `SelectFromModel` | Single fit, threshold filter | Faster (1 model fit) | Good -- but misses feature interactions |

**Practical advice:** Start with `SelectFromModel` for a quick baseline. If your feature set is manageable ($p < 100$) and you need higher accuracy, switch to `RFECV()`, which combines RFE with cross-validation to find the optimal number of features automatically.

Feature engineering is as much art as science. Domain knowledge often suggests transformations that no automated method would discover. A data scientist who understands their domain can engineer features that dramatically outperform any black-box feature selection -- this is why understanding the *business context* is at least as important as knowing the *algorithms*.

## Choosing the Right Preprocessing Technique

The decision tree below summarizes how to select the appropriate data transformation based on your data's characteristics. This is Figure 2.17 from the textbook.

**START: What is your data's primary characteristic?**

**Branch 1 -- Has Outliers?**
- YES $\rightarrow$ `RobustScaler()`, `QuantileTransformer()`, or `PowerTransformer()`
- NO $\rightarrow$ Continue to Distribution Type

**Branch 2 -- Distribution Type?**
- Normal $\rightarrow$ `StandardScaler()` or `MinMaxScaler()`
- Skewed $\rightarrow$ `PowerTransformer()` (Yeo-Johnson or Box-Cox)
- Unknown/Mixed $\rightarrow$ `QuantileTransformer()` or `Normalizer()`
- Sparse $\rightarrow$ `MaxAbsScaler()` or `Normalizer()`

**Branch 3 -- Special Requirements?**
- Need $[0, 1]$ range $\rightarrow$ `MinMaxScaler()`
- Need unit norm $\rightarrow$ `Normalizer()`
- Preserve zeros (sparse data) $\rightarrow$ `MaxAbsScaler()`
- Custom range $\rightarrow$ `MinMaxScaler(feature_range=...)`

The key insight: **there is no universal "best" scaler**. The choice depends on your data distribution, the algorithm you plan to use, and any specific constraints on the output range.

## Chapter Summary

This chapter covered the essential preprocessing techniques that prepare raw data for ML modeling:

| Technique | Tool | Purpose |
|:----------|:-----|:--------|
| **Missing Data** | `SimpleImputer`, `KNNImputer`, `IterativeImputer` | Fill gaps with estimated values |
| **Scaling** | `StandardScaler`, `MinMaxScaler`, `Normalizer` | Put features on comparable scales |
| **Encoding** | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` | Convert categories to numbers |
| **Pipelines** | `Pipeline` | Chain preprocessing + modeling, prevent leakage |
| **Feature Creation** | `PolynomialFeatures`, `KBinsDiscretizer` | Generate new informative features |
| **Feature Selection** | `RFE`, `SelectFromModel` | Remove uninformative features |

**Three rules to live by:**

1. **Always split before transforming.** Call `train_test_split()` first, then `fit_transform()` on training data and `transform()` on test data. Never the reverse.

2. **Use pipelines.** Every preprocessing step that learns from data (`fit`) must be inside a pipeline to prevent leakage. This includes imputation, scaling, encoding, and feature selection.

3. **Not all models need all preprocessing.** Tree-based models (Decision Trees, Random Forests) are invariant to feature scaling and can handle ordinal encodings naturally. Distance-based models (KNN, SVM) and gradient-based models (linear/logistic regression, neural networks) require careful preprocessing.
